# 50K + Fuzzy Features + LoRA — Cross-Encoder on Colab T4

**Runtime:** T4 GPU. Then `Run All`.

Fuzzy features (rapidfuzz name/addr ratios, JaroWinkler, Metaphone jaccard, house-number jaccard) are computed per pair and prepended to the model input as `[nm=.. jw=.. ad=.. ph=.. nu=..]`. The model conditions its same/different decision on both text and those signals.

Data assumed at `/MyDrive/amazon-entity-resolution/dataset/train/train_labeled_pairs.tsv`.

Expected wall time on T4: ~25-45 min total (setup ~2 min, sampling ~2 min, 50K training ~15 min, full-val eval ~15-25 min).

## 1. GPU + Drive + deps

In [ ]:
!nvidia-smi | head -20
from google.colab import drive
drive.mount('/content/drive')

# torchao pin: Colab pre-installs an old version that breaks peft's dispatcher.
!pip install -q "transformers>=4.44.0" "peft>=0.11.0" \
    "accelerate>=0.33.0" "torchao>=0.16.0" \
    rapidfuzz jellyfish sentencepiece protobuf

DRIVE = '/content/drive/MyDrive/amazon-entity-resolution'
DATA  = '/content/drive/MyDrive/amazon-entity-resolution/dataset/train'
import os
os.makedirs(DRIVE, exist_ok=True)
!ls -lh {DATA}

## 2. Write sampler

In [ ]:
%%writefile /content/sample_labeled_pairs.py
"""Build stratified 100k / 500k train subsets + a shared val split.

Sampling is by source1 entity (not by row), so all pairs belonging to one
source1 land in the same split — no leakage across train/val. The 1:3
positive:negative ratio from the full file is preserved automatically because
we take all rows for every sampled source1.

Val is 10% of source1 ids, held out before either train subset is sampled and
identical for both, so the 100k and 500k runs are directly comparable.

Usage:
    python sample_labeled_pairs.py \\
        --input  dataset/train/train_labeled_pairs.tsv \\
        --outdir dataset/train
"""
import argparse
import csv
import random
import sys
from pathlib import Path

csv.field_size_limit(sys.maxsize)

SEED = 42
VAL_FRAC = 0.10
TRAIN_SIZES = {"100k": 100_000, "500k": 500_000}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--outdir", required=True, type=Path)
    args = ap.parse_args()

    print(f"Pass 1: collecting source1 ids and row counts from {args.input.name} ...", flush=True)
    src1_rows: dict[str, int] = {}
    with args.input.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            s1 = row["source1_entity_id"]
            src1_rows[s1] = src1_rows.get(s1, 0) + 1
    print(f"  {len(src1_rows):,} distinct source1 entities", flush=True)

    rng = random.Random(SEED)
    all_ids = list(src1_rows.keys())
    rng.shuffle(all_ids)
    n_val = int(len(all_ids) * VAL_FRAC)
    val_ids = set(all_ids[:n_val])
    train_pool = all_ids[n_val:]
    print(f"  val: {len(val_ids):,} source1 ids", flush=True)

    picks: dict[str, set[str]] = {}
    for tag, target_rows in TRAIN_SIZES.items():
        chosen: set[str] = set()
        total = 0
        for s1 in train_pool:
            if total >= target_rows:
                break
            chosen.add(s1)
            total += src1_rows[s1]
        picks[tag] = chosen
        print(f"  train_{tag}: {len(chosen):,} source1 ids -> {total:,} rows", flush=True)

    outs = {
        "val": args.outdir / "val_pairs.tsv",
        "100k": args.outdir / "train_pairs_100k.tsv",
        "500k": args.outdir / "train_pairs_500k.tsv",
    }
    print("Pass 2: writing splits ...", flush=True)
    with args.input.open("r", encoding="utf-8", newline="") as f, \
            outs["val"].open("w", encoding="utf-8", newline="") as fv, \
            outs["100k"].open("w", encoding="utf-8", newline="") as f100, \
            outs["500k"].open("w", encoding="utf-8", newline="") as f500:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        for w in (fv, f100, f500):
            csv.writer(w, delimiter="\t", lineterminator="\n").writerow(header)
        wv = csv.writer(fv, delimiter="\t", lineterminator="\n")
        w100 = csv.writer(f100, delimiter="\t", lineterminator="\n")
        w500 = csv.writer(f500, delimiter="\t", lineterminator="\n")
        counts = {"val": 0, "100k": 0, "500k": 0}
        for row in reader:
            s1 = row[0]
            if s1 in val_ids:
                wv.writerow(row); counts["val"] += 1
            if s1 in picks["100k"]:
                w100.writerow(row); counts["100k"] += 1
            if s1 in picks["500k"]:
                w500.writerow(row); counts["500k"] += 1
    for tag, n in counts.items():
        print(f"  {outs[tag].name}: {n:,} rows", flush=True)


if __name__ == "__main__":
    main()


## 3. Write fuzzy training script

In [ ]:
%%writefile /content/train_cross_encoder_fuzzy.py
"""LoRA fine-tune mDeBERTa-v3 cross-encoder with fuzzy features prepended.

Same skeleton as train_cross_encoder.py. Difference: per-pair rapidfuzz +
jellyfish scores are computed on the fly in the Dataset and inserted as a
bracket-tagged prefix in the "Listing A" segment:

    [nm=85 jw=90 ad=72 ph=65 nu=50]
    Listing A: <s1_name> | <s1_addr>
    Listing B: <cand_name> | <cand_addr>

Fields:
    nm  = rapidfuzz token_set_ratio(name_a, name_b)         [0-100]
    jw  = JaroWinkler(squash(name_a), squash(name_b)) * 100 [0-100]
    ad  = rapidfuzz token_set_ratio(addr_a, addr_b)         [0-100]
    ph  = jaccard(metaphone tokens of name_a vs name_b)     [0-100]
    nu  = jaccard(digit tokens of addr_a vs addr_b)         [0-100]

Usage:
    python train_cross_encoder_fuzzy.py \\
        --train dataset/train/train_pairs_100k.tsv \\
        --val   dataset/train/val_pairs.tsv \\
        --outdir models/cross_encoder_50k_fuzzy \\
        --max-train 50000 --epochs 3 --batch 32
"""
import argparse
import csv
import json
import os
import re
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np  # noqa: E402
import torch  # noqa: E402
from torch.utils.data import Dataset  # noqa: E402
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType  # noqa: E402
from rapidfuzz import fuzz  # noqa: E402
from rapidfuzz.distance import JaroWinkler  # noqa: E402
import jellyfish  # noqa: E402

csv.field_size_limit(sys.maxsize)

BASE_MODEL = "microsoft/mdeberta-v3-base"
MAX_LEN = 224  # a few extra tokens over the vanilla script for the fuzzy prefix

_WS = re.compile(r"\s+")
_NUM = re.compile(r"\b\d+\b")


def squash(s: str) -> str:
    return _WS.sub("", s.lower())


def _phon_jaccard(a: str, b: str) -> int:
    ca = {jellyfish.metaphone(w) for w in a.lower().split() if w}
    cb = {jellyfish.metaphone(w) for w in b.lower().split() if w}
    ca.discard(""); cb.discard("")
    if not ca or not cb:
        return 0
    return int(round(100 * len(ca & cb) / len(ca | cb)))


def _num_jaccard(a: str, b: str) -> int:
    na = set(_NUM.findall(a or ""))
    nb = set(_NUM.findall(b or ""))
    if not na or not nb:
        return 0
    return int(round(100 * len(na & nb) / len(na | nb)))


def fuzzy_features(name_a: str, addr_a: str, name_b: str, addr_b: str) -> dict:
    return {
        "nm": int(round(fuzz.token_set_ratio(name_a or "", name_b or ""))),
        "jw": int(round(100 * JaroWinkler.normalized_similarity(squash(name_a or ""), squash(name_b or "")))),
        "ad": int(round(fuzz.token_set_ratio(addr_a or "", addr_b or ""))),
        "ph": _phon_jaccard(name_a or "", name_b or ""),
        "nu": _num_jaccard(addr_a or "", addr_b or ""),
    }


def record_text(name: str, addr: str) -> str:
    return f"{name} | {addr or 'no address'}"


def build_pair_text(row: dict) -> tuple[str, str]:
    f = fuzzy_features(
        row["source1_business_name"], row["source1_business_address"],
        row["candidate_business_name"], row["candidate_business_address"],
    )
    prefix = f"[nm={f['nm']} jw={f['jw']} ad={f['ad']} ph={f['ph']} nu={f['nu']}]"
    a = f"{prefix} Listing A: {record_text(row['source1_business_name'], row['source1_business_address'])}"
    b = f"Listing B: {record_text(row['candidate_business_name'], row['candidate_business_address'])}"
    return a, b


class PairDataset(Dataset):
    def __init__(self, path: Path, tokenizer, max_len: int = MAX_LEN, max_rows: int | None = None):
        self.rows: list[dict] = []
        with path.open("r", encoding="utf-8", newline="") as f:
            for i, r in enumerate(csv.DictReader(f, delimiter="\t")):
                if max_rows is not None and i >= max_rows:
                    break
                self.rows.append(r)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        r = self.rows[idx]
        a, b = build_pair_text(r)
        enc = self.tok(a, b, truncation=True, max_length=self.max_len, padding=False)
        enc["labels"] = int(r["label"])
        return enc


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-(logits[:, 1] - logits[:, 0])))
    preds = (probs >= 0.5).astype(np.int64)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f05 = (1.25 * precision * recall / (0.25 * precision + recall)) if (precision + recall) else 0.0
    acc = float((preds == labels).mean())
    return {"accuracy": acc, "precision": precision, "recall": recall, "f0_5": f05}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", required=True, type=Path)
    ap.add_argument("--val", required=True, type=Path)
    ap.add_argument("--outdir", required=True, type=Path)
    ap.add_argument("--base-model", default=BASE_MODEL)
    ap.add_argument("--max-train", type=int, default=None,
                    help="Cap the number of training rows read (from top of file).")
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--lora-dropout", type=float, default=0.1)
    ap.add_argument("--warmup-ratio", type=float, default=0.05)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--eval-limit", type=int, default=10_000)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
    print(f"device: {device}", flush=True)

    print(f"loading base model: {args.base_model}", flush=True)
    tok = AutoTokenizer.from_pretrained(args.base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.base_model,
        num_labels=2,
        id2label={0: "different_business", 1: "same_business"},
        label2id={"different_business": 0, "same_business": 1},
    )

    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        target_modules=["query_proj", "key_proj", "value_proj"],
        bias="none",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    print(f"loading train: {args.train}  (max_rows={args.max_train})", flush=True)
    train_ds = PairDataset(args.train, tok, max_rows=args.max_train)
    print(f"  train pairs: {len(train_ds):,}", flush=True)
    print(f"loading val:   {args.val}", flush=True)
    val_ds = PairDataset(args.val, tok)
    if args.eval_limit and len(val_ds) > args.eval_limit:
        rng = np.random.default_rng(args.seed)
        idx = rng.choice(len(val_ds), size=args.eval_limit, replace=False)
        val_ds = torch.utils.data.Subset(val_ds, sorted(idx.tolist()))
    print(f"  val pairs (in-training eval): {len(val_ds):,}", flush=True)

    use_fp16 = device == "cuda"
    training_args = TrainingArguments(
        output_dir=str(args.outdir / "checkpoints"),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch,
        per_device_eval_batch_size=args.batch * 2,
        learning_rate=args.lr,
        warmup_ratio=args.warmup_ratio,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=50,
        report_to="none",
        seed=args.seed,
        fp16=use_fp16,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        metric_for_best_model="f0_5",
        greater_is_better=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tok,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
    )

    print("training ...", flush=True)
    trainer.train()

    print(f"saving adapter + tokenizer to {args.outdir}", flush=True)
    model.save_pretrained(args.outdir)
    tok.save_pretrained(args.outdir)
    (args.outdir / "training_args.json").write_text(json.dumps(vars(args), default=str, indent=2))
    metrics = trainer.evaluate()
    (args.outdir / "final_val_metrics.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


## 4. Write fuzzy eval script

In [ ]:
%%writefile /content/eval_cross_encoder_fuzzy.py
"""Evaluate the fuzzy-augmented LoRA cross-encoder on val_pairs.tsv.

Must use the SAME input format as train_cross_encoder_fuzzy.py — the model
saw fuzzy features in a bracket prefix during training, so eval must too.

Usage:
    python eval_cross_encoder_fuzzy.py \\
        --adapter models/cross_encoder_50k_fuzzy \\
        --val     dataset/train/val_pairs.tsv \\
        --out     dataset/train/eval_cross_50k_fuzzy.json
"""
import argparse
import csv
import json
import os
import re
import sys
import time
from collections import defaultdict
from pathlib import Path

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np  # noqa: E402
import torch  # noqa: E402
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
from peft import PeftModel  # noqa: E402
from rapidfuzz import fuzz  # noqa: E402
from rapidfuzz.distance import JaroWinkler  # noqa: E402
import jellyfish  # noqa: E402

csv.field_size_limit(sys.maxsize)

MAX_LEN = 224

_WS = re.compile(r"\s+")
_NUM = re.compile(r"\b\d+\b")


def squash(s: str) -> str:
    return _WS.sub("", s.lower())


def _phon_jaccard(a: str, b: str) -> int:
    ca = {jellyfish.metaphone(w) for w in a.lower().split() if w}
    cb = {jellyfish.metaphone(w) for w in b.lower().split() if w}
    ca.discard(""); cb.discard("")
    if not ca or not cb:
        return 0
    return int(round(100 * len(ca & cb) / len(ca | cb)))


def _num_jaccard(a: str, b: str) -> int:
    na = set(_NUM.findall(a or ""))
    nb = set(_NUM.findall(b or ""))
    if not na or not nb:
        return 0
    return int(round(100 * len(na & nb) / len(na | nb)))


def build_pair_text(row: dict) -> tuple[str, str]:
    na = row["source1_business_name"]; aa = row["source1_business_address"]
    nb = row["candidate_business_name"]; ab = row["candidate_business_address"]
    nm = int(round(fuzz.token_set_ratio(na or "", nb or "")))
    jw = int(round(100 * JaroWinkler.normalized_similarity(squash(na or ""), squash(nb or ""))))
    ad = int(round(fuzz.token_set_ratio(aa or "", ab or "")))
    ph = _phon_jaccard(na or "", nb or "")
    nu = _num_jaccard(aa or "", ab or "")
    prefix = f"[nm={nm} jw={jw} ad={ad} ph={ph} nu={nu}]"
    a = f"{prefix} Listing A: {na} | {aa or 'no address'}"
    b = f"Listing B: {nb} | {ab or 'no address'}"
    return a, b


def sweep_thresholds(scores, labels):
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    tp = fp = 0
    P = sum(labels)
    best = (0.5, 0.0, 0.0, 0.0)
    for idx in order:
        if labels[idx] == 1: tp += 1
        else: fp += 1
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / P if P else 0.0
        if precision + recall == 0: continue
        f05 = 1.25 * precision * recall / (0.25 * precision + recall)
        if f05 > best[3]:
            best = (float(scores[idx]), precision, recall, f05)
    return {"threshold": best[0], "precision": best[1], "recall": best[2], "f0_5": best[3]}


def fixed_threshold(scores, labels, thr):
    tp = fp = fn = 0
    for s, y in zip(scores, labels):
        pred = 1 if s >= thr else 0
        if pred == 1 and y == 1: tp += 1
        elif pred == 1 and y == 0: fp += 1
        elif pred == 0 and y == 1: fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    if precision + recall == 0:
        return {"threshold": thr, "precision": 0.0, "recall": 0.0, "f0_5": 0.0}
    f05 = 1.25 * precision * recall / (0.25 * precision + recall)
    return {"threshold": thr, "precision": precision, "recall": recall, "f0_5": f05}


def source1_macro_f05(pair_records, scores, thr):
    truth_by_s1: dict[str, set[str]] = defaultdict(set)
    preds_by_s1: dict[str, set[str]] = defaultdict(set)
    all_s1: set[str] = set()
    for rec, s in zip(pair_records, scores):
        s1 = rec["source1_entity_id"]
        cid = rec["candidate_entity_id"]
        all_s1.add(s1)
        if int(rec["label"]) == 1:
            truth_by_s1[s1].add(cid)
        if s >= thr:
            preds_by_s1[s1].add(cid)
    f05s = []
    for s1 in all_s1:
        pred = preds_by_s1.get(s1, set())
        true = truth_by_s1.get(s1, set())
        if not true and not pred:
            f05s.append(1.0); continue
        if not pred:
            f05s.append(0.0); continue
        tp = len(pred & true)
        precision = tp / len(pred) if pred else 0.0
        recall = tp / len(true) if true else 0.0
        if precision + recall == 0:
            f05s.append(0.0); continue
        f05s.append(1.25 * precision * recall / (0.25 * precision + recall))
    return sum(f05s) / len(f05s) if f05s else 0.0


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter", required=True, type=Path)
    ap.add_argument("--base", default="microsoft/mdeberta-v3-base")
    ap.add_argument("--val", required=True, type=Path)
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"device: {device}", flush=True)

    print(f"loading base: {args.base}", flush=True)
    tok = AutoTokenizer.from_pretrained(str(args.adapter))
    base = AutoModelForSequenceClassification.from_pretrained(
        args.base,
        num_labels=2,
        id2label={0: "different_business", 1: "same_business"},
        label2id={"different_business": 0, "same_business": 1},
    )
    print(f"loading adapter: {args.adapter}", flush=True)
    model = PeftModel.from_pretrained(base, str(args.adapter))
    model.eval().to(device)

    print(f"loading val: {args.val}", flush=True)
    pair_records: list[dict] = []
    with args.val.open("r", encoding="utf-8", newline="") as f:
        for i, r in enumerate(csv.DictReader(f, delimiter="\t")):
            if args.limit is not None and i >= args.limit:
                break
            pair_records.append(r)
    print(f"  val pairs: {len(pair_records):,}", flush=True)

    scores: list[float] = []
    t0 = time.time()
    B = args.batch
    with torch.no_grad():
        for i in range(0, len(pair_records), B):
            chunk = pair_records[i:i + B]
            texts = [build_pair_text(r) for r in chunk]
            enc = tok([t[0] for t in texts], [t[1] for t in texts],
                      truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits.detach().float().cpu().numpy()
            probs = 1.0 / (1.0 + np.exp(-(logits[:, 1] - logits[:, 0])))
            scores.extend(float(p) for p in probs)
            if (i // B) % 20 == 0:
                done = i + len(chunk)
                speed = done / max(time.time() - t0, 1e-6)
                print(f"  scored {done:,}/{len(pair_records):,} ({speed:.1f} pair/s)", flush=True)
    dur = time.time() - t0
    print(f"scoring done in {dur/60:.1f} min ({len(pair_records)/dur:.1f} pair/s).", flush=True)

    labels = [int(r["label"]) for r in pair_records]
    metrics = {
        "pair_at_0.5": fixed_threshold(scores, labels, 0.5),
        "pair_best_threshold": sweep_thresholds(scores, labels),
    }
    for thr in (0.3, 0.5, 0.7, metrics["pair_best_threshold"]["threshold"]):
        metrics[f"source1_macro_f0_5_at_{thr:.3f}"] = source1_macro_f05(pair_records, scores, thr)

    result = {
        "adapter": str(args.adapter),
        "base": args.base,
        "val": str(args.val),
        "n_pairs": len(pair_records),
        "n_pos": sum(labels),
        "n_neg": len(labels) - sum(labels),
        "n_source1_in_val": len({r["source1_entity_id"] for r in pair_records}),
        "seconds_total": dur,
        "pairs_per_second": len(pair_records) / dur if dur else 0.0,
        "metrics": metrics,
    }
    args.out.write_text(json.dumps(result, indent=2))
    print(f"wrote {args.out}", flush=True)
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


## 5. Sample: create val + train_pairs_100k on Drive (~2 min)

Sampler builds 100K and 500K subsets; we cap the training at 50K via `--max-train 50000` in cell 6. Seed=42 so results are reproducible.

In [ ]:
!python /content/sample_labeled_pairs.py \
    --input  {DATA}/train_labeled_pairs.tsv \
    --outdir {DATA}
!ls -lh {DATA}/val_pairs.tsv {DATA}/train_pairs_100k.tsv

## 6. Train — 50K, fuzzy features, LoRA, fp16, batch 32 (~15 min on T4)

In [ ]:
!python /content/train_cross_encoder_fuzzy.py \
    --train  {DATA}/train_pairs_100k.tsv \
    --val    {DATA}/val_pairs.tsv \
    --outdir {DRIVE}/cross_encoder_50k_fuzzy \
    --max-train 50000 \
    --epochs 3 --batch 32 --lr 1e-4

## 7. Full-val eval (~15-25 min on T4)

In [ ]:
!python /content/eval_cross_encoder_fuzzy.py \
    --adapter {DRIVE}/cross_encoder_50k_fuzzy \
    --val     {DATA}/val_pairs.tsv \
    --out     {DRIVE}/eval_cross_50k_fuzzy.json \
    --batch   128
!cat {DRIVE}/eval_cross_50k_fuzzy.json

## 8. What's saved on Drive

```
amazon-entity-resolution/
├── dataset/train/
│   ├── val_pairs.tsv
│   ├── train_pairs_100k.tsv
│   └── train_pairs_500k.tsv
├── cross_encoder_50k_fuzzy/
│   ├── adapter_model.safetensors    (~10-50 MB LoRA weights)
│   ├── adapter_config.json
│   ├── tokenizer files
│   ├── training_args.json
│   └── final_val_metrics.json
└── eval_cross_50k_fuzzy.json
```

Metric to compare against the M3's baseline: `metrics.source1_macro_f0_5_at_0.500`.